# Detecção Hierárquica de Toxicidade no Civil Comments
## CNN + Bi-LSTM com seleção *nested* de thresholds

**Notebook final de pesquisa**

Este notebook documenta a versão final adotada para a entrega acadêmica do projeto. O sistema usa uma arquitetura paralela **CNN + Bi-LSTM** e uma estratégia hierárquica em dois estágios para classificação de toxicidade no dataset **Civil Comments**.

### Resultado principal

O benchmark científico adotado como referência é o resultado completo da execução de validação cruzada hierárquica com seleção *nested* de thresholds:

| Avaliação | Macro F1 |
|---|---:|
| End-to-end com thresholds fixos | 0.3496 |
| **End-to-end com thresholds nested** | **0.4412** |
| Stage 2 oracle com thresholds nested | 0.5959 |

Uma repetição independente do mesmo protocolo produziu **0.4427 Macro F1 end-to-end**, reforçando que a região de desempenho observada é aproximadamente **0.44**, e não resultado de uma única execução favorável.

> O valor `0.4412` é mantido como benchmark principal porque corresponde ao experimento controlado e documentado da PR #10. O `0.4427` é tratado como evidência de repetição, não como substituição oportunista pelo maior número observado.

## 1. Pergunta de pesquisa

A questão investigada é:

> Uma arquitetura híbrida CNN + Bi-LSTM, combinada com uma estratégia hierárquica e seleção de thresholds exclusivamente na validação interna, consegue classificar subtipos de toxicidade de forma metodologicamente válida no Civil Comments?

O trabalho não busca afirmar estado da arte. O objetivo é construir e avaliar de forma reprodutível um sistema de NLP com:

- preservação de labels sobrepostos;
- isolamento entre treino, validação interna e avaliação externa;
- avaliação separada de roteamento e classificação;
- medição explícita da propagação de erro da cascata;
- seleção de thresholds sem usar o fold externo.

## 2. Dataset e targets

**Dataset:** `google/civil_comments`

O split de treino utilizado contém **1.804.874 comentários**.

### Stage 1 — roteamento

O Stage 1 aprende:

- `toxicity` — target contínuo/fracionário;
- `severe_toxicity` — saída auxiliar contínua.

A definição de verdade de referência para encaminhar um comentário ao Stage 2 é:

```text
toxicity >= 0.4
```

Essa escolha foi baseada em análise completa de cobertura do dataset.

### Stage 2 — classificação multilabel

Para os comentários roteados, o Stage 2 prediz independentemente:

- `obscene`
- `threat`
- `insult`
- `identity_attack`
- `sexual_explicit`

Os targets são preservados como valores fracionários durante o treinamento, e a avaliação binária usa `score >= 0.5`.

## 3. Cobertura do gate de verdade de referência

Com `toxicity >= 0.4`:

| Estatística | Valor |
|---|---:|
| Comentários roteados | 201.476 |
| Proporção do treino | 11,16% |
| Exemplos com ao menos um label Stage 2 positivo | 126.250 |
| Positivos Stage 2 perdidos pelo gate | 533 |
| Cobertura de qualquer label Stage 2 positivo | **99,578%** |

Cobertura por subtipo:

| Label | Positivos perdidos |
|---|---:|
| obscene | 0,219% |
| threat | 0,304% |
| insult | 0,128% |
| identity_attack | 1,037% |
| sexual_explicit | 4,823% |

`sexual_explicit` é a principal limitação conhecida da definição de gate.

## 4. Arquitetura

O encoder mantém a ideia central do experimento original:

```text
Texto
  ↓
Tokenização / sequências
  ↓
Embedding treinável
  ├───────────────┬────────────────┐
  ↓               ↓                │
Bi-LSTM       Conv1D multi-kernel   │
  ↓               ↓                │
Pooling / representação             │
  └───────────────┴────────────────┘
                  ↓
             Concatenação
                  ↓
                Dense
                  ↓
              Sigmoids
```

A arquitetura é aplicada separadamente aos dois estágios.

### Por que dois estágios?

O Stage 1 responde:

> este comentário deve entrar na classificação detalhada?

O Stage 2 responde:

> quais subtipos de toxicidade estão presentes?

Isso permite medir separadamente o desempenho do roteamento e da classificação multilabel.

## 5. Protocolo de validação

O experimento final usa **2 folds externos**.

Em cada fold:

1. o fold externo é reservado exclusivamente para avaliação;
2. o restante é dividido em treino e validação interna;
3. tokenização e transformações aprendidas são ajustadas apenas no treino;
4. `EarlyStopping` usa somente a validação interna;
5. thresholds do Stage 2 são escolhidos somente na validação interna;
6. o threshold de roteamento do Stage 1 é escolhido pela **Macro F1 end-to-end da validação interna**;
7. todos os thresholds são congelados antes da avaliação externa.

### Importante

O fold externo **não participa** de:

- treinamento;
- `EarlyStopping`;
- escolha de threshold;
- seleção de modelo.

Essa separação é a principal correção metodológica em relação ao notebook histórico.

## 6. Resultado final — Stage 1

### Threshold fixo vs. threshold selecionado na validação interna

| Métrica | Fixo `0.40` | Nested |
|---|---:|---:|
| Accuracy | 0.9257 | 0.9265 |
| Precision | 0.8340 | 0.7163 |
| Recall | 0.4178 | **0.5653** |
| F1 | 0.5567 | **0.6319** |
| Routing rate | 0.0559 | 0.0881 |
| PR-AUC / AP | 0.7095 | 0.7095 |
| ROC-AUC | 0.9195 | 0.9195 |

### Interpretação

O ranking do modelo não mudou — PR-AUC e ROC-AUC são os mesmos — mas a decisão binária melhorou substancialmente.

O threshold fixo era conservador demais:

- tinha maior precisão;
- encaminhava poucos comentários;
- perdia muitos positivos.

A seleção nested aceitou mais falsos positivos para recuperar mais comentários realmente relevantes, elevando o recall de `0.4178` para `0.5653`.

## 7. Resultado final — Stage 2 oracle

Nesta avaliação, o Stage 2 recebe **somente os exemplos que realmente deveriam ser roteados**.

| Label | F1 fixo | F1 nested |
|---|---:|---:|
| obscene | 0.5294 | **0.6177** |
| threat | 0.4572 | **0.5112** |
| insult | 0.7486 | **0.7639** |
| identity_attack | 0.2841 | **0.5595** |
| sexual_explicit | 0.3635 | **0.5272** |
| **Macro F1** | **0.4766** | **0.5959** |

O ganho de `identity_attack` é particularmente relevante: o modelo já possuía sinal discriminativo, mas o threshold fixo de `0.5` estava inadequado para a distribuição de scores da classe.

> `Stage 2 oracle` não é o desempenho do sistema completo. Ele mede o classificador multilabel supondo roteamento perfeito.

## 8. Resultado final — sistema end-to-end

Esta é a avaliação principal do sistema completo:

```text
comentário → Stage 1 → Stage 2 → labels finais
```

| Label | F1 fixo | F1 nested |
|---|---:|---:|
| obscene | 0.4608 | **0.5115** |
| threat | 0.1991 | **0.2951** |
| insult | 0.6327 | **0.6343** |
| identity_attack | 0.1758 | **0.3761** |
| sexual_explicit | 0.2797 | **0.3892** |
| **Macro F1** | **0.3496** | **0.4412** |

A melhoria absoluta foi:

```text
0.4412 - 0.3496 = 0.0916
```

equivalente a aproximadamente **26% de melhoria relativa** no mesmo conjunto de modelos treinados.

A melhoria ocorreu sem substituir a arquitetura principal e sem introduzir múltiplas técnicas simultaneamente. O principal ganho veio da escolha adequada do ponto operacional de decisão.

## 9. Repetição do experimento

Uma segunda execução completa do mesmo protocolo produziu:

| Métrica | Repetição |
|---|---:|
| Stage 1 tuned F1 | 0.6351 |
| Stage 1 tuned recall | 0.5780 |
| Stage 1 PR-AUC | 0.7086 |
| Stage 1 ROC-AUC | 0.9195 |
| Stage 2 oracle Macro F1 | 0.5967 |
| **End-to-end Macro F1** | **0.4427** |

A proximidade entre:

```text
0.4412
0.4427
```

é usada como evidência prática de estabilidade da região de desempenho do pipeline.

O benchmark principal permanece `0.4412` para evitar selecionar retroativamente a execução com o maior valor.

## 10. Visualização dos resultados validados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.DataFrame(
    {
        "Fixed": [0.5567, 0.4766, 0.3496],
        "Nested": [0.6319, 0.5959, 0.4412],
    },
    index=["Stage 1 F1", "Stage 2 Oracle Macro F1", "End-to-End Macro F1"],
)

display(summary)

ax = summary.plot(kind="bar", figsize=(9, 5))
ax.set_ylim(0, 0.75)
ax.set_ylabel("F1")
ax.set_title("Threshold fixo vs. seleção nested")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 11. Preparar ambiente de reprodução

O bloco abaixo funciona em Google Colab e também serve como referência para ambientes locais.

Ele clona o repositório se necessário e, se ele já existir, atualiza `main` apenas quando não existem alterações locais.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Umbura/Hatespeech_Detection_Civil_Comments_NLP_Obsolete.git"
REPO_DIR = Path("/content/Hatespeech_Detection_Civil_Comments_NLP_Obsolete")

if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        status = subprocess.check_output(
            ["git", "status", "--porcelain"],
            cwd=REPO_DIR,
            text=True,
        ).strip()
        if status:
            raise RuntimeError(
                "O clone possui alterações locais. "
                "A atualização automática foi interrompida para preservar os arquivos."
            )

        subprocess.run(["git", "fetch", "origin", "--prune"], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "switch", "main"], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)

    os.chdir(REPO_DIR)
    print("Repository:", REPO_DIR)
    print(
        "Commit:",
        subprocess.check_output(
            ["git", "log", "-1", "--oneline"],
            cwd=REPO_DIR,
            text=True,
        ).strip(),
    )
else:
    print("Ambiente local detectado. Execute o notebook a partir da raiz do repositório.")

### Dependências

No Colab, para reproduzir o ambiente usado nas execuções completas registradas, use:

In [ ]:
%pip install -q \
  tensorflow==2.20.0 \
  datasets==5.0.0 \
  iterative-stratification==0.1.9 \
  imbalanced-learn==0.14.2

## 12. Verificar cobertura do gate

Esta etapa não treina o modelo e pode ser usada para confirmar a definição hierárquica no dataset completo.

In [ ]:
!python scripts/analyze_gate_coverage.py

## 13. Smoke test

Use o smoke somente para verificar se o pipeline executa corretamente.

**Não reporte essas métricas como resultado científico.**

In [ ]:
!python scripts/run_hierarchical_cv.py \
    --max-samples 50000 \
    --n-splits 2 \
    --epochs 1

## 14. Reprodução completa

A execução abaixo corresponde ao protocolo final:

```text
2 folds externos
até 5 épocas por estágio
EarlyStopping
seleção nested de thresholds
dataset completo
```

Em CPU pode ser lenta. GPU é recomendada.

In [ ]:
!python scripts/run_hierarchical_cv.py \
    --n-splits 2 \
    --epochs 5

## 15. Limitações

O resultado final deve ser interpretado com as seguintes limitações:

1. o projeto utiliza 2 folds externos devido ao custo computacional;
2. a arquitetura começa a apresentar overfitting em épocas iniciais;
3. ainda existe propagação de erro entre Stage 1 e Stage 2;
4. `sexual_explicit` é a categoria mais afetada pelo gate de verdade de referência;
5. fairness e robustez por subgrupos não foram avaliadas;
6. não há alegação de estado da arte;
7. o benchmark final deste repositório é uma estimativa por validação cruzada, não uma avaliação congelada no split oficial de teste;
8. o notebook histórico contém limitações metodológicas e não deve ser usado como benchmark atual.

Essas limitações fazem parte do resultado científico e não devem ser ocultadas.

## 16. Conclusão

A versão final do experimento mostra que a arquitetura CNN + Bi-LSTM possui capacidade discriminativa relevante no Civil Comments, mas que thresholds fixos de decisão eram inadequados para o problema desbalanceado e hierárquico.

A seleção nested:

- elevou o Stage 1 F1 para `0.6319`;
- elevou o Stage 2 oracle Macro F1 para `0.5959`;
- elevou o desempenho end-to-end de `0.3496` para **`0.4412` Macro F1**;
- apresentou uma repetição independente em **`0.4427`**.

A principal conclusão não é que o sistema seja estado da arte, mas que **a escolha metodologicamente correta dos thresholds recupera uma parcela substancial do desempenho sem alterar a arquitetura**, e que a cascata ainda apresenta um gargalo mensurável de roteamento.

Para a entrega acadêmica, o projeto é considerado concluído nesta configuração.

## 17. Referências principais

- Google / Jigsaw. **Civil Comments** dataset.
- Pitsilis, G. K. *Improved two-stage hate speech classification for Twitter based on Deep Neural Networks*. 2022.
- Zhou, C. et al. *A C-LSTM Neural Network for Text Classification*. COLING, 2016.
- Schuster, M.; Paliwal, K. K. *Bidirectional recurrent neural networks*. IEEE Transactions on Signal Processing, 1997.

Consulte também:

- `docs/TARGET_STRATEGY.md`
- `docs/EXPERIMENT_HISTORY.md`
- `docs/REPRODUCIBILITY.md`
- `results/FINAL_RESULTS.md`